In [ ]:
library(tidyverse)
library(ggplot2)
library(dplyr)

system("gsutil cp $calculated_pc_cloud_location .")

## Plot PCs

In [ ]:
# umap{X} suffix to files refers to how many rounds of UMAP filtering/PC recalculation were done 
# final set includes afr_umap1, amr_umap0, eur_umap2

In [ ]:
# AFR

#svd_afr <- read.csv('svd_afr_umap1.csv', sep='t')
#svd_afr[] <- lapply(svd_afr, function(x) as.numeric(gsub("[/]", "", x)))
#write.table(svd_afr, "cleaned_afr_umap1_pcs.csv", sep='\t', quote=F, row.names=F)
svd_afr <- read.csv('cleaned_afr_umap1_pcs.csv', sep='\t')

# view first 10 PCs
par(mfrow=c(3,3))

for (i in 1:9) {
    count=i
    plot(svd_afr[,i] ~ svd_afr[,i+1], pch=16, xlab=i+1, ylab=i, col='brown3')
    count=+2
}

In [ ]:
# AMR

#svd_amr <- read.csv('svd_amr_umap0.csv', sep='t')
#svd_amr[] <- lapply(svd_amr, function(x) as.numeric(gsub("[/]", "", x)))
#write.table(svd_amr, "cleaned_amr_umap0_pcs.csv", sep='\t', quote=F, row.names=F)
svd_amr <- read.csv('cleaned_amr_umap0_pcs.csv', sep='\t')

par(mfrow=c(3,3))

for (i in 1:9) {
    count=i
    plot(svd_amr[,i] ~ svd_amr[,i+1], pch=16, xlab=i+1, ylab=i, col='chartreuse3')
    count=+2
}

In [ ]:
# EUR

#svd_eur <- read.csv('svd_eur_umap2.csv', sep='t')
#svd_eur[] <- lapply(svd_eur, function(x) as.numeric(gsub("[/]", "", x)))
#write.table(svd_eur, "cleaned_eur_umap2_pcs.csv", sep='\t', quote=F, row.names=F)
svd_eur <- read.csv('cleaned_eur_umap2_pcs.csv', sep='\t')

par(mfrow=c(3,3))

for (i in 1:9) {
    count=i
    plot(svd_eur[,i] ~ svd_eur[,i+1], pch=16, xlab=i+1, ylab=i, col='cyan3')
    count=+2
}

## Outlier sample detection / UMAP plots

In [ ]:
#install.packages("devtools")
#devtools::install_github("jlmelville/uwot")
library(uwot)
library(ggplot2)
library(dplyr)

### AFR filtering

In [41]:
umap_afr <- data.frame(umap(svd_afr[,1:4], min_dist=0,n_threads=4))
# PCs (1:7, 8:12) for round 0
# 1:4 for round 1 

In [ ]:
# get imputed fam for ID list
afrIDs <- read.table("keep_umap1_afr_fidiid.txt", sep='')
# check that number of individuals is the same
dim(afrIDs) == dim(umap_afr)

In [ ]:
# add individual IDs to umap df
umap_afr$person_id <- afrIDs[,2]
umap_afr$fam_id <- umap_afr$person_id
head(umap_afr)

In [ ]:
# check UMAP distribution before filtering
ggplot(data=umap_afr, aes(x = umap_afr[,1], y = umap_afr[,2])) +
    geom_point(size = 1, alpha = 0.3, color='chartreuse3') +
    labs(
    title = paste("AFR PCs pre-UMAP filtering")) +
  theme_minimal()

In [ ]:
# filter
umap_afr1 <- filter(umap_afr, 
                    umap_afr[,2]< 5 & umap_afr[,2] > (-1), .preserve=T)

umap_afr1 <- filter(umap_afr1, 
                    umap_afr1[,1] < (2), .preserve=T)

In [ ]:
# check restricted individuals
plot(umap_afr[,1:2])
axis(1, -15:10)
axis(2, -10:15)
points(umap_afr1[,c(1:2)], col='brown3')

In [ ]:
# write file of individuals to keep (use for next PC calculation round)
write.table(umap_afr1[,3:4], "keep_umap1_afr_fidiid.txt", quote=F, row.names=F, col.names=F)
write.table(umap_afr, "umap1_afr_results.txt", quote=F, row.names=F, col.names=F)

### AMR filtering
Note: unfiltered AMR data was used for the final analysis

In [ ]:
umap_amr <- data.frame(umap(svd_amr[,c(1:13, 15:22, 33:40, 46:50, 54:62, 82:90, 91:95, 97:108)], 
                            min_dist=0,n_threads=4))
#PCs 1:13, 15:22, 33:40, 46:50, 54:62, 82:90, 91:95, 97:108 for round 0. based on PC plot above

In [ ]:
amrIDs <- read.table("amrQC.postimputation.fam", sep='')

# check that number of individuals is the same
dim(amrIDs) == dim(umap_amr)

# add individual IDs to umap df
umap_amr$person_id <- amrIDs[,1]
umap_amr$fam_id <- amrIDs[,2]
head(umap_amr)

In [ ]:
# check UMAP distribution before filtering
ggplot(data=umap_amr, aes(x = umap_amr[,1], y = umap_amr[,2])) +
    geom_point(size = 1, alpha = 0.3, color='chartreuse3') +
    labs(
    title = paste("AMR pre-UMAP filtering")) +
  theme_minimal()

In [16]:
umap_amr2 <- filter(umap_amr, 
                    umap_amr[,2]< 5 & umap_amr[,2] > (-1), .preserve=T)

umap_amr2 <- filter(umap_amr2, 
                    umap_amr2[,1] < (2), .preserve=T)

In [ ]:
# check restricted individuals
plot(umap_amr[,c(1:2)])
points(umap_amr2[,c(1:2)], col='chartreuse3', main="AMR UMAP0 vs UMAP1")

In [ ]:
# write file of individuals to keep 
write.table(umap_amr2[,3:4], "keep_umap1_amr_fidiid.txt", quote=F, row.names=F, col.names=F)
write.table(umap_amr, "umap0_amr_results.txt", quote=F, row.names=F, col.names=F)

### EUR filtering

In [49]:
umap_eur <- data.frame(umap(svd_eur[,1:6], min_dist=0, n_threads=4))
# PCs 1-12 for base PCs
# PCs 1-6 for umap1
# PCs 1-4 for umap2

In [ ]:
eurIDs <- read.table("eurQC.postimputation.fam", sep='')

# check that number of individuals is the same
dim(eurIDs) == dim(umap_eur)

# add individual IDs to umap df
umap_eur$person_id <- eurIDs[,1]
umap_eur$fam_id <- eurIDs[,2]
head(umap_eur)

In [ ]:
# check UMAP distribution before filtering
ggplot(data=umap_eur, aes(x = umap_eur[,1], y = umap_eur[,2])) +
    geom_point(size = 1, alpha = 0.2, color='cyan3') +
    labs(
    title = paste("EUR pre-UMAP filtering")) +
  theme_minimal()

In [ ]:
# filter
umap_eur1 <- filter(umap_eur, 
                    umap_eur[,2] <4 & umap_eur[,2] > (-8), .preserve=T)
umap_eur1 <- filter(umap_eur1, 
                    umap_eur1[,1] > -3.7  & umap_eur1[,1] < 8, .preserve=T)

In [ ]:
# plot PC clusters

ggplot(data=umap_eur1, aes(x = umap_eur[,1], y = umap_eur[,2])) +
    geom_point(size = 1, alpha = 0.2, color='cyan3') +
    labs(
    title = "EUR post-UMAP filtering") +
  theme_minimal()

In [ ]:
# write file of individuals to keep 
write.table(umap_eur2[,3:4], "keep_umap1_eur_fidiid.txt", quote=F, row.names=F, col.names=F)
write.table(umap_eur, "umap1_eur_results.txt", quote=F, row.names=F, col.names=F)

In [ ]:
# EUR UMAP round 2
# after loading in recalculated PCs for cohort above to recalculate UMAP. this resultant dataframe is <eur>

# Define circle parameters
X1_center <- -2
X2_center <- -1
r <- 4

umap_eur3 <- eur %>%
  mutate(distance = sqrt((X1 - X1_center)^2 + (X2 - X2_center)^2)) %>%
  filter(distance <= r)

# check distribution
plot(umap_eur3$X1, umap_eur3$X2)
dim(umap_eur3)
dim(eur)

In [ ]:
# write file of individuals to keep 
write.table(umap_eur3[,3:4], "keep_umap2_eur_fidiid.txt", quote=F, row.names=F, col.names=F)
write.table(umap_eur2, "umap2_eur_results.txt", quote=F, row.names=F, col.names=F)

# Investigating self-reported ethnicity x ancestry UMAP
For visualisation only

In [ ]:
# read in ethnicity survey data
ethn <- read.table('aou.ethnicities', sep='\t', header=T)
head(ethn)

In [ ]:
# read in UMAPs
umap_afr <- read.table('umap1_afr_results.txt', sep='', header=T)
colnames(umap_afr) <- c("X1", "X2", "IID", "FID")
umap_amr <- read.table('umap0_amr_results.txt', sep='', header=T)
colnames(umap_amr) <- c("X1", "X2", "IID", "FID")
umap_eur <- read.table('umap2_eur_results.txt', sep='', header=T)
colnames(umap_eur) <- c("X1", "X2", "IID", "FID")

# add ethnicity answers
labeled_umap_afr <- merge(umap_afr[,1:3], ethn[,c('IID', 'answer')], by = 'IID', all.x=T)
labeled_umap_amr <- merge(umap_amr[,1:3], ethn[,c('IID', 'answer')], by = 'IID', all.x=T)
labeled_umap_eur <- merge(umap_eur[,1:3], ethn[,c('IID', 'answer')], by = 'IID', all.x=T)

In [ ]:
# AFR
labeled_umap_afr$answer[grep("AIAN",labeled_umap_afr$answer)] <- 'AIAN/Hispanic: any'
labeled_umap_afr$answer[grep("Hispanic",labeled_umap_afr$answer)] <- 'AIAN/Hispanic: any'
labeled_umap_afr$answer[grep("Black None",labeled_umap_afr$answer)] <- 'Black: Other'
labeled_umap_afr$answer[grep("Asian",labeled_umap_afr$answer)] <- 'Asian Specific: any'
labeled_umap_afr$answer[grep("White",labeled_umap_afr$answer)] <- 'White Specific: any'
labeled_umap_afr$answer[grep("Skip",labeled_umap_afr$answer)] <- NA

labeled_umap_afr$answer<-sub("Black Specific: African American", "Black Specific: African \n American", labeled_umap_afr$answer) 
labeled_umap_afr$answer<-sub("Black Specific: Barbadian", "Black Specific: Caribbean", labeled_umap_afr$answer) 
labeled_umap_afr$answer<-sub("Black Specific: Haitian", "Black Specific: Caribbean", labeled_umap_afr$answer) 
labeled_umap_afr$answer<-sub("Black Specific: Jamaican", "Black Specific: Caribbean", labeled_umap_afr$answer) 
labeled_umap_afr$answer<-sub("Black Specific: Liberian", "Black Specific: West African", labeled_umap_afr$answer) 
labeled_umap_afr$answer<-sub("Black Specific: Nigerian", "Black Specific: West African", labeled_umap_afr$answer) 
labeled_umap_afr$answer<-sub("Black Specific: Ghanaian", "Black Specific: West African", labeled_umap_afr$answer) 
labeled_umap_afr$answer<-sub("Black Specific: Ethiopian", "Black Specific: East African", labeled_umap_afr$answer) 
labeled_umap_afr$answer<-sub("Black Specific: Somali", "Black Specific: East African", labeled_umap_afr$answer) 

In [ ]:
# AMR
labeled_umap_amr$answer[grep("AIAN None",labeled_umap_amr$answer)] <- 'AIAN: other'
labeled_umap_amr$answer<-sub("AIAN Specific: American Indian", "AIAN Specific: American \n Indian", labeled_umap_amr$answer) 
labeled_umap_amr$answer<-sub("AIAN Specific: Central South American Indian", "AIAN Specific: American \n Indian", labeled_umap_amr$answer) 
labeled_umap_amr$answer[grep("Hispanic None",labeled_umap_amr$answer)] <- 'Hispanic: other'
labeled_umap_amr$answer[grep("Spanish",labeled_umap_amr$answer)] <- 'Hispanic: other'
labeled_umap_amr$answer[grep("Black",labeled_umap_amr$answer)] <- 'Black Specific: any'
labeled_umap_amr$answer[grep("Asian",labeled_umap_amr$answer)] <- 'Asian Specific: any'
labeled_umap_amr$answer[grep("White",labeled_umap_amr$answer)] <- 'White Specific: any'
labeled_umap_amr$answer[grep("Skip",labeled_umap_amr$answer)] <- NA

labeled_umap_amr$answer<-sub("Hispanic Specific: Puerto Rican", "Hispanic Specific: Caribbean", labeled_umap_amr$answer) 
labeled_umap_amr$answer<-sub("Hispanic Specific: Dominican", "Hispanic Specific: Caribbean", labeled_umap_amr$answer) 
labeled_umap_amr$answer<-sub("Hispanic Specific: Cuban", "Hispanic Specific: Caribbean", labeled_umap_amr$answer) 
labeled_umap_amr$answer<-sub("Hispanic Specific: Mexican", "Hispanic Specific: Central American", labeled_umap_amr$answer) 
labeled_umap_amr$answer<-sub("Hispanic Specific: Honduran", "Hispanic Specific: Central American", labeled_umap_amr$answer) 
labeled_umap_amr$answer<-sub("Hispanic Specific: Salvadoran", "Hispanic Specific: Central American", labeled_umap_amr$answer) 
labeled_umap_amr$answer<-sub("Hispanic Specific: Colombian", "Hispanic Specific: South American", labeled_umap_amr$answer) 
labeled_umap_amr$answer<-sub("Hispanic Specific: Ecuadorian", "Hispanic Specific: South American", labeled_umap_amr$answer) 

In [ ]:
# EUR
labeled_umap_eur$answer[grep("AIAN",labeled_umap_eur$answer)] <- 'AIAN'
labeled_umap_eur$answer[grep("Hispanic",labeled_umap_eur$answer)] <- 'Hispanic Specific: any'
labeled_umap_eur$answer[grep("Black",labeled_umap_eur$answer)] <- 'Black Specific: any'
labeled_umap_eur$answer[grep("Asian",labeled_umap_eur$answer)] <- 'Asian Specific: any'
labeled_umap_eur$answer[grep("European",labeled_umap_eur$answer)] <- 'White: other'
labeled_umap_eur$answer[grep("Skip",labeled_umap_eur$answer)] <- NA
labeled_umap_eur$answer[grep("White None",labeled_umap_eur$answer)] <- 'White: other'

labeled_umap_eur$answer<-sub("White Specific: Norwegian", "White Specific: \n North European", labeled_umap_eur$answer) 
labeled_umap_eur$answer<-sub("White Specific: Italian", "White Specific: \n South European", labeled_umap_eur$answer) 
labeled_umap_eur$answer<-sub("White Specific: Spanish", "White Specific: \n South European", labeled_umap_eur$answer) 
labeled_umap_eur$answer<-sub("White Specific: English", "White Specific: \n British Isles", labeled_umap_eur$answer)
labeled_umap_eur$answer<-sub("White Specific: Irish", "White Specific: \n British Isles", labeled_umap_eur$answer) 
labeled_umap_eur$answer<-sub("White Specific: Scottish", "White Specific: \n British Isles", labeled_umap_eur$answer) 
labeled_umap_eur$answer<-sub("White Specific: Polish", "White Specific: \n Central European", labeled_umap_eur$answer) 
labeled_umap_eur$answer<-sub("White Specific: German", "White Specific: \n West European", labeled_umap_eur$answer) 
labeled_umap_eur$answer<-sub("White Specific: Dutch", "White Specific: \n West European", labeled_umap_eur$answer) 
labeled_umap_eur$answer<-sub("White Specific: French", "White Specific: \n West European", labeled_umap_eur$answer) 

In [ ]:
table(labeled_umap_df$answer) # frequency per unique entry in self-reported ethnicity survey 

In [ ]:
afr_plot <- ggplot(data=labeled_umap_afr,  aes(x = UMAP_1, y = UMAP_2, colour=answer)) +
    geom_point(data=filter(labeled_umap_afr, is.na(labeled_umap_afr$answer)), size = 2, alpha=0.5) +
    geom_point(data=filter(labeled_umap_afr, labeled_umap_afr$answer == 'White Specific: any'), size = 2, alpha=0.5) +
    geom_jitter(width = 0.2, height=0.2, 
                data=filter(labeled_umap_afr, !is.na(labeled_umap_afr$answer) & labeled_umap_afr$answer != 'White Specific: any'), 
                size = 2, alpha=0.7) +
    labs(
    title = "UMAP of 120 PCs for individuals with African-like genetically-inferred ancestry", 
    color = 'Self-reported ethnicity') + 
scale_colour_manual(values = c('pink', 'orange', "yellow", "yellow3", "purple4", "red2", "yellow2", "darkgoldenrod4", "coral","coral3", "purple1", "purple4", "red3", "green2")) +
theme_minimal(base_size=14) 

afr_plot

In [ ]:
amr_plot <- ggplot(data=labeled_umap_amr,  aes(x = UMAP_1, y = UMAP_2, colour=answer)) +
    geom_point(data=filter(labeled_umap_amr, is.na(labeled_umap_amr$answer)), size = 2, alpha=0.5) +
    geom_point(data=filter(labeled_umap_amr, labeled_umap_amr$answer == 'White Specific: any'), size = 2, alpha=0.5) +
    geom_jitter(width = 0.2, height=0.2, 
                data=filter(labeled_umap_amr, !is.na(labeled_umap_amr$answer) & labeled_umap_amr$answer != 'White Specific: any'), 
                size = 2, alpha=0.7) +
    labs(
    title = "UMAP of 120 PCs for individuals with Admixed American-like genetically-inferred ancestry", 
    color = 'Self-reported ethnicity') + 
scale_colour_manual(values = c("darkgoldenrod1", "yellow", "yellow3", "chocolate", "pink", "red", "darkcyan", "blue", "cyan3", "cyan", "darkgreen", "green2", "steelblue", "yellowgreen", "purple4", "purple1", "purple4")) +
theme_minimal(base_size=14) 

amr_plot

In [ ]:
eur_plot <- ggplot(data=labeled_umap_eur,  aes(x = UMAP_1, y = UMAP_2, colour=answer)) +
    geom_point(data=filter(labeled_umap_eur, is.na(labeled_umap_eur$answer)), size = 2, alpha=0.5) +
    geom_jitter(width = 0.2, height=0.2, 
                data=filter(labeled_umap_eur, !is.na(labeled_umap_eur$answer), size = 2, alpha=0.7) +
    labs(
    title = "UMAP of 120 PCs for individuals with European-like genetically-inferred ancestry", 
    color = 'Self-reported ethnicity') + 
scale_colour_manual(values = c("black", "black", "black", "red", "orangered", "purple1", "yellow4", "yellow", "purple3",
                       "red4", "blue", "cyan", "purple4", "red", "grey80")) +
    theme_minimal(base_size=14) 

eur_plot

Number of self-reported ethnicities per individual

In [ ]:
freqs <- data.frame(sort(table(afr$IID)))
colnames(freqs) <- c('IID', 'ancestry_freq')

freqs <- merge(freqs, afr, by='IID')
#freqs <- arrange(freqs$ancestry_freq)
head(freqs)

In [ ]:
table(freqs$ancestry_freq)
table(filter(freqs, freqs$answer=='Black Specific: African American')$ancestry_freq)

In [ ]:
lazyfilter <- distinct(freqs, FID, .keep_all = TRUE)
dim(freqs)
dim(lazyfilter)

In [ ]:
lazyfilter$answer[grep("Hispanic",lazyfilter$answer)] <- 'Hispanic'
lazyfilter$answer[grep("Black",lazyfilter$answer)] <- 'Black'
lazyfilter$answer[grep("Asian",lazyfilter$answer)] <- 'Asian'
lazyfilter$answer[grep("Skip",lazyfilter$answer)] <- NA
lazyfilter$answer[grep("None",lazyfilter$answer)] <- NA

In [ ]:
# visualise number of self-reported ethnicities by UMAP plot
# to check if there is a pattern between high levels of genetically-predicted population admixture/stratification and high levels of self-reported admixture

freqs$ancestry_freq <- as.numeric(freqs$ancestry_freq)
freqs <- arrange(freqs, ancestry_freq)

ggplot(data=freqs,  aes(x = X1, y = X2, colour=ancestry_freq)) +
    geom_point(shape='+', size=3) +
    labs(
    title = "Self-reported ethnicities for AFR individuals", 
    color = 'Num. ancestries reported', 
    x = "PC 1",
    y = "PC 2",) + scale_color_gradient2(midpoint=20, low="blue", mid="cyan", high="red", space ="Lab" ) +
theme_minimal()